# Career Guidance System - Complete Model Training

This notebook contains all model training code for the Career Guidance System.

## Models to Train:
1. **Content-Based Recommendation Model** - Uses TF-IDF for job/role recommendations
2. **Collaborative Filtering Model** - Uses SVD for user-item recommendations
3. **Career Role Classifier** - Random Forest classifier for role prediction
4. **Chatbot Intent Classifier** - Logistic Regression for intent classification

## Instructions:
1. **Setup Kaggle API** (Step 2): Enter your API token (KGAT_... format)
2. **Download Datasets** (Step 4): Downloads real datasets from Kaggle
3. **Preprocess Data** (Step 5): Processes raw datasets for training
4. **Train Models** (Steps 6-9): Trains all ML models
5. **Download Results**: Download trained models and datasets

## Required Datasets from Kaggle:
- Resume Dataset: `snehaanbhawal/resume-dataset`
- IT Jobs Market: `asaniczka/it-jobs-market-analysis-2023`
- Data Science Jobs: `andrewmvd/data-science-jobs`
- LinkedIn Job Postings: `arshkon/linkedin-job-postings`

## Step 1: Install Dependencies


In [ ]:
# Install required packages including Kaggle API
# Note: NumPy 1.26.4 is required for compatibility with 'surprise' library
# IMPORTANT: If you get binary incompatibility errors, RESTART RUNTIME after running this cell

print("📦 Installing packages (this may take 2-3 minutes)...")
print("   ⚠️  Note: Some dependency warnings are expected and can be ignored\n")

# Step 1: Uninstall NumPy 2.x if present, then install NumPy 1.26.4 specifically
print("   Step 1: Installing NumPy 1.26.4 (required for 'surprise' library)...")
!pip uninstall numpy -y -q 2>/dev/null || true
!pip install "numpy==1.26.4" --no-cache-dir -q

# Step 2: Install pandas (will try to upgrade NumPy, but we'll fix it)
print("   Step 2: Installing pandas...")
!pip install "pandas<2.1" --no-cache-dir -q
# Re-pin NumPy after pandas installation
!pip install "numpy==1.26.4" --force-reinstall --no-deps -q

# Step 3: Install scipy
print("   Step 3: Installing scipy...")
!pip install scipy --no-cache-dir -q
# Re-pin NumPy after scipy installation
!pip install "numpy==1.26.4" --force-reinstall --no-deps -q

# Step 4: Install scikit-learn
print("   Step 4: Installing scikit-learn...")
!pip install scikit-learn --no-cache-dir -q
# Re-pin NumPy after scikit-learn installation
!pip install "numpy==1.26.4" --force-reinstall --no-deps -q

# Step 5: Install remaining packages (these don't require NumPy 2.x)
print("   Step 5: Installing remaining packages...")
!pip install joblib sentence-transformers kaggle -q

# Step 6: Install surprise (this is the one that needs NumPy < 2.0)
print("   Step 6: Installing surprise (collaborative filtering library)...")
!pip install surprise --no-cache-dir -q

# Step 7: Final check - ensure NumPy is still 1.26.4
print("   Step 7: Final NumPy version check...")
!pip install "numpy==1.26.4" --force-reinstall --no-deps -q

# Verify installation
print("\n✅ Verifying installation...")
try:
    import numpy as np
    import pandas as pd
    import sklearn
    from surprise import Dataset, Reader
    
    np_version = np.__version__
    if np_version.startswith('1.'):
        print(f"   ✅ NumPy: {np_version} (compatible with 'surprise')")
    else:
        print(f"   ⚠️  NumPy: {np_version} (should be 1.x - restart runtime and re-run)")
    
    print(f"   ✅ Pandas: {pd.__version__}")
    print(f"   ✅ Scikit-learn: {sklearn.__version__}")
    print(f"   ✅ Surprise: Installed")
    print("\n✅ All packages installed successfully!")
    print("\n⚠️  Note: Dependency warnings about opencv/jax/tensorflow are expected")
    print("   These packages are pre-installed in Colab but won't affect our models")
except Exception as e:
    print(f"   ⚠️  Error during verification: {e}")
    print("\n💡 Solution:")
    print("   1. Go to: Runtime → Restart runtime")
    print("   2. Re-run this cell")
    print("   3. The packages will be reinstalled with correct binary compatibility")


## Step 2: Setup Kaggle API

**Important:** You need Kaggle API credentials to download datasets.

### How to Get Your Credentials:
1. **Get your username:**
   - Go to https://www.kaggle.com/settings
   - Your username is shown at the top of the page (e.g., "dineshadhikari9860")

2. **Get your API token:**
   - On the same page, scroll to "API" section
   - Click "Generate New Token" or "Create New API Token"
   - Copy the token (starts with `KGAT_...`)
   - **Important:** Copy it immediately - you won't see it again!

3. **Enter both in the cell below:**
   - Username: Your Kaggle username
   - Token: Your API token

### Option A: Upload kaggle.json file
If you downloaded `kaggle.json`, you can upload it instead (uncomment Option A in the code cell).

### Option B: Enter Credentials Manually
Enter your username and API token in the code cell below.

In [ ]:
# Setup Kaggle API
import os
from pathlib import Path
import json

# Create .kaggle directory
os.makedirs('/root/.kaggle', exist_ok=True)

# Option A: Upload kaggle.json file (if you have it)
# Uncomment the lines below to upload kaggle.json:
# from google.colab import files
# uploaded = files.upload()
# for fn in uploaded.keys():
#     if fn == 'kaggle.json':
#         !cp kaggle.json /root/.kaggle/
#         !chmod 600 /root/.kaggle/kaggle.json
#         print("✅ Kaggle API configured from uploaded file")

# Option B: Enter API Token and Username (NEW FORMAT - KGAT_...)
# Paste your API token from Kaggle Settings → API → Generate New Token
# Example token format: KGAT_38a6b85087cfdcac127739f7de3b2597
KAGGLE_API_TOKEN = "YOUR_KAGGLE_API_TOKEN_HERE"  # ⚠️ Replace with YOUR token from Kaggle
KAGGLE_USERNAME = "YOUR_KAGGLE_USERNAME_HERE"     # ⚠️ Replace with YOUR Kaggle username

# Set environment variable (for new token format)
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

# Create kaggle.json with both username and token (required by Kaggle API client)
if KAGGLE_API_TOKEN != "YOUR_KAGGLE_API_TOKEN_HERE" and KAGGLE_USERNAME != "YOUR_KAGGLE_USERNAME_HERE":
    kaggle_config = {
        "username": KAGGLE_USERNAME,
        "key": KAGGLE_API_TOKEN
    }
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump(kaggle_config, f)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    
    print("✅ Kaggle API configured successfully!")
    print(f"   Username: {KAGGLE_USERNAME}")
    print(f"   Token: {KAGGLE_API_TOKEN[:10]}...{KAGGLE_API_TOKEN[-5:]}")
else:
    print("⚠️  Please configure your Kaggle credentials:")
    print("   1. Replace 'YOUR_KAGGLE_API_TOKEN_HERE' with your API token")
    print("   2. Replace 'YOUR_KAGGLE_USERNAME_HERE' with your Kaggle username")
    print("   Get your token from: https://www.kaggle.com/settings → API → Generate New Token")
    print("   Your username is shown on: https://www.kaggle.com/settings (top of page)")

## Step 3: Import Libraries and Setup Directories


In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import json
import re
import zipfile
import shutil
from collections import Counter
import ast

# ML Libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Try to import optional libraries
try:
    from surprise import Dataset, Reader, SVD as SurpriseSVD
    from surprise.model_selection import train_test_split as surprise_train_test_split
    from surprise import accuracy
    SURPRISE_AVAILABLE = True
except ImportError:
    SURPRISE_AVAILABLE = False
    print("⚠️ Surprise library not available, using sklearn TruncatedSVD")

try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_BERT_AVAILABLE = True
except ImportError:
    SENTENCE_BERT_AVAILABLE = False
    print("⚠️ Sentence-BERT not available, using TF-IDF only")

# Create necessary directories
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('ml/models', exist_ok=True)

print("✅ All libraries imported and directories created!")


## Step 4: Download Datasets from Kaggle


In [ ]:
# Download datasets from Kaggle
print("="*70)
print("DOWNLOADING DATASETS FROM KAGGLE")
print("="*70)

# Define datasets to download
datasets = [
    {
        'name': 'resume-dataset',
        'kaggle_path': 'snehaanbhawal/resume-dataset',
        'description': 'Resume dataset with skills and experience'
    },
    {
        'name': 'it-jobs-market',
        'kaggle_path': 'asaniczka/it-jobs-market-analysis-2023',
        'description': 'IT jobs market analysis data'
    },
    {
        'name': 'data-science-jobs',
        'kaggle_path': 'andrewmvd/data-science-jobs',
        'description': 'Data science job postings'
    },
    {
        'name': 'linkedin-job-postings',
        'kaggle_path': 'arshkon/linkedin-job-postings',
        'description': 'LinkedIn job postings'
    }
]

# Download each dataset
for dataset in datasets:
    print(f"\n📥 Downloading {dataset['name']}...")
    try:
        !kaggle datasets download -d {dataset['kaggle_path']} -p data/raw/{dataset['name']} --unzip
        print(f"✅ {dataset['name']} downloaded successfully")
    except Exception as e:
        print(f"⚠️ Error downloading {dataset['name']}: {e}")
        print(f"   You may need to accept terms on Kaggle: https://www.kaggle.com/datasets/{dataset['kaggle_path']}")

print("\n" + "="*70)
print("✅ Dataset download complete!")
print("="*70)


## Step 5: Preprocess Datasets

Process raw Kaggle datasets into the format needed for training.


In [ ]:
# Data Preprocessing Class
class DataPreprocessor:
    """Preprocess datasets for career guidance system"""
    
    def __init__(self, raw_data_dir='data/raw', processed_data_dir='data/processed'):
        self.raw_data_dir = Path(raw_data_dir)
        self.processed_data_dir = Path(processed_data_dir)
        self.processed_data_dir.mkdir(parents=True, exist_ok=True)
    
    def _clean_skills(self, skills_str):
        """Clean and normalize skills string"""
        if pd.isna(skills_str):
            return []
        
        skills_str = str(skills_str).lower()
        skills = re.split(r'[,;|]|\s+and\s+', skills_str)
        
        cleaned = []
        for skill in skills:
            skill = skill.strip()
            skill = re.sub(r'^(proficient in|expert in|knowledge of|experience with)\s+', '', skill)
            skill = skill.strip()
            if len(skill) > 2:
                cleaned.append(skill)
        
        return list(set(cleaned))
    
    def _clean_text(self, text):
        """Clean text description"""
        if pd.isna(text):
            return ""
        text = str(text)
        text = re.sub(r'<[^>]+>', '', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    
    def _extract_years(self, exp_str):
        """Extract years of experience from string"""
        if pd.isna(exp_str):
            return 0
        exp_str = str(exp_str).lower()
        match = re.search(r'(\d+)\s*\+?\s*years?', exp_str)
        if match:
            return int(match.group(1))
        return 0
    
    def _normalize_role(self, role):
        """Normalize role/job title"""
        if pd.isna(role):
            return ""
        role = str(role).lower().strip()
        role = re.sub(r'\s+', ' ', role)
        return role.title()
    
    def preprocess_resume_dataset(self):
        """Preprocess resume/user profile datasets"""
        print("Preprocessing resume datasets...")
        
        # Look for resume CSV files
        resume_files = []
        resume_dirs = ['resume-dataset', 'it-jobs-market']
        
        for dir_name in resume_dirs:
            dir_path = self.raw_data_dir / dir_name
            if dir_path.exists():
                csv_files = list(dir_path.glob('*.csv'))
                resume_files.extend(csv_files)
        
        if not resume_files:
            print("⚠️ No resume CSV files found")
            return None
        
        # Read and combine all resume files
        dfs = []
        for file in resume_files:
            try:
                df = pd.read_csv(file)
                dfs.append(df)
                print(f"  Loaded {file.name}: {len(df)} rows")
            except Exception as e:
                print(f"  ⚠️ Error reading {file.name}: {e}")
        
        if not dfs:
            return None
        
        combined_df = pd.concat(dfs, ignore_index=True)
        print(f"  Combined dataset: {len(combined_df)} rows")
        
        # Clean and standardize
        # Handle skills - might be string or list
        if 'skills' in combined_df.columns:
            # Convert string representations of lists to actual lists
            def parse_skills(s):
                if pd.isna(s):
                    return []
                if isinstance(s, list):
                    return s
                s = str(s)
                # Try to parse as Python list
                try:
                    import ast
                    parsed = ast.literal_eval(s)
                    if isinstance(parsed, list):
                        return parsed
                except:
                    pass
                # Otherwise treat as string and split
                return self._clean_skills(s)
            
            combined_df['skills_cleaned'] = combined_df['skills'].apply(parse_skills)
            combined_df['skills_cleaned'] = combined_df['skills'].apply(self._clean_skills)
        else:
            skill_cols = [col for col in combined_df.columns if 'skill' in col.lower()]
            if skill_cols:
                combined_df['skills_cleaned'] = combined_df[skill_cols].apply(
                    lambda row: self._clean_skills(' '.join(row.astype(str))), axis=1
                )
        
        if 'experience' in combined_df.columns:
            combined_df['experience_years'] = combined_df['experience'].apply(self._extract_years)
        elif 'experience_years' not in combined_df.columns:
            combined_df['experience_years'] = 0
        
        if 'role' in combined_df.columns or 'job_title' in combined_df.columns:
            role_col = 'role' if 'role' in combined_df.columns else 'job_title'
            combined_df['current_role'] = combined_df[role_col].apply(self._normalize_role)
        elif 'current_role' not in combined_df.columns:
            combined_df['current_role'] = 'Unknown'
        
        # Add ID if missing
        if 'id' not in combined_df.columns:
            combined_df['id'] = range(len(combined_df))
        
        # Save processed data
        output_path = self.processed_data_dir / 'resumes_processed.csv'
        combined_df.to_csv(output_path, index=False)
        print(f"✅ Saved processed resume data: {output_path} ({len(combined_df)} rows)")
        return combined_df
    
    def preprocess_job_postings(self):
        """Preprocess job postings datasets"""
        print("Preprocessing job postings datasets...")
        
        # Look for job CSV files
        job_files = []
        job_dirs = ['data-science-jobs', 'linkedin-job-postings']
        
        for dir_name in job_dirs:
            dir_path = self.raw_data_dir / dir_name
            if dir_path.exists():
                csv_files = list(dir_path.glob('*.csv'))
                job_files.extend(csv_files)
        
        if not job_files:
            print("⚠️ No job CSV files found")
            return None
        
        # Read and combine all job files
        dfs = []
        for file in job_files:
            try:
                df = pd.read_csv(file)
                dfs.append(df)
                print(f"  Loaded {file.name}: {len(df)} rows")
            except Exception as e:
                print(f"  ⚠️ Error reading {file.name}: {e}")
        
        if not dfs:
            return None
        
        combined_df = pd.concat(dfs, ignore_index=True)
        print(f"  Combined dataset: {len(combined_df)} rows")
        
        # Clean and standardize
        if 'description' in combined_df.columns:
            combined_df['description_cleaned'] = combined_df['description'].apply(self._clean_text)
        
        if 'required_skills' in combined_df.columns or 'skills' in combined_df.columns:
            skill_col = 'required_skills' if 'required_skills' in combined_df.columns else 'skills'
            combined_df['skills_cleaned'] = combined_df[skill_col].apply(self._clean_skills)
        elif 'description' in combined_df.columns:
            # Extract skills from description (simplified)
            combined_df['skills_cleaned'] = combined_df['description'].apply(self._clean_skills)
        
        if 'title' in combined_df.columns or 'job_title' in combined_df.columns:
            title_col = 'title' if 'title' in combined_df.columns else 'job_title'
            combined_df['title'] = combined_df[title_col]
        
        # Add ID if missing
        if 'id' not in combined_df.columns:
            combined_df['id'] = range(len(combined_df))
        
        # Save processed data
        output_path = self.processed_data_dir / 'jobs_processed.csv'
        combined_df.to_csv(output_path, index=False)
        print(f"✅ Saved processed job data: {output_path} ({len(combined_df)} rows)")
        return combined_df
    
    def preprocess_interactions(self):
        """Create interactions dataset from available data"""
        print("Creating interactions dataset...")
        
        # Try to find existing interactions file
        interaction_files = list(self.raw_data_dir.glob('**/interactions*.csv'))
        interaction_files.extend(list(self.raw_data_dir.glob('**/*interaction*.csv')))
        
        if interaction_files:
            df = pd.read_csv(interaction_files[0])
            print(f"  Loaded existing interactions: {len(df)} rows")
        else:
            # Create sample interactions from resumes and jobs
            print("  No interactions file found, creating sample interactions...")
            resumes_df = pd.read_csv(self.processed_data_dir / 'resumes_processed.csv') if (self.processed_data_dir / 'resumes_processed.csv').exists() else None
            jobs_df = pd.read_csv(self.processed_data_dir / 'jobs_processed.csv') if (self.processed_data_dir / 'jobs_processed.csv').exists() else None
            
            if resumes_df is not None and jobs_df is not None:
                n_users = min(len(resumes_df), 200)
                n_jobs = min(len(jobs_df), 500)
                n_interactions = min(2000, n_users * n_jobs // 2)
                
                interactions = []
                for _ in range(n_interactions):
                    user_id = np.random.randint(0, n_users)
                    job_id = np.random.randint(0, n_jobs)
                    rating = np.random.choice([3, 4, 5], p=[0.2, 0.4, 0.4]) if np.random.random() < 0.7 else np.random.choice([1, 2, 3])
                    interactions.append({'user_id': user_id, 'item_id': job_id, 'rating': rating})
                
                df = pd.DataFrame(interactions)
                df = df.drop_duplicates(subset=['user_id', 'item_id'])
                print(f"  Created {len(df)} sample interactions")
            else:
                print("  ⚠️ Cannot create interactions - missing resume or job data")
                return None
        
        # Normalize IDs
        if 'user_id' in df.columns:
            df['user_id'] = pd.Categorical(df['user_id']).codes
        if 'item_id' in df.columns or 'job_id' in df.columns:
            item_col = 'item_id' if 'item_id' in df.columns else 'job_id'
            df['item_id'] = pd.Categorical(df[item_col]).codes
        
        if 'rating' in df.columns:
            df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
            df['rating'] = df['rating'].fillna(df['rating'].median()).astype(int)
            df['rating'] = df['rating'].clip(1, 5)
        else:
            df['rating'] = 1
        
        output_path = self.processed_data_dir / 'interactions_processed.csv'
        df.to_csv(output_path, index=False)
        print(f"✅ Saved processed interactions: {output_path} ({len(df)} rows)")
        return df

# Run preprocessing
print("="*70)
print("PREPROCESSING DATASETS")
print("="*70)

preprocessor = DataPreprocessor()
resumes_df = preprocessor.preprocess_resume_dataset()
jobs_df = preprocessor.preprocess_job_postings()
interactions_df = preprocessor.preprocess_interactions()

print("\n" + "="*70)
print("✅ Data preprocessing complete!")
print("="*70)


## Step 3: Content-Based Recommendation Model


In [ ]:
# Content-Based Recommender Class
class ContentBasedRecommender:
    """Content-based recommendation using TF-IDF and/or Sentence-BERT"""
    
    def __init__(self, use_sbert=False):
        self.use_sbert = use_sbert and SENTENCE_BERT_AVAILABLE
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words='english',
            min_df=2,
            max_df=0.95
        )
        self.sbert_model = None
        if self.use_sbert:
            print("Loading Sentence-BERT model...")
            self.sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
        
        self.user_vectors = None
        self.job_vectors = None
        self.role_vectors = None
        self.user_ids = None
        self.job_ids = None
        self.role_ids = None
    
    def prepare_user_features(self, users_df, fit_vectorizer=False):
        """Prepare user feature vectors"""
        print("Preparing user features...")
        
        user_texts = []
        user_ids = []
        
        for _, user in users_df.iterrows():
            features = []
            
            if 'skills_cleaned' in user and isinstance(user['skills_cleaned'], list):
                features.extend(user['skills_cleaned'])
            elif 'skills' in user:
                skills = str(user['skills']).lower()
                features.extend(skills.split())
            
            if 'experience_years' in user:
                features.append(f"{int(user['experience_years'])} years experience")
            
            if 'current_role' in user and pd.notna(user['current_role']):
                features.append(str(user['current_role']).lower())
            
            if 'interests' in user and isinstance(user['interests'], list):
                features.extend([str(i).lower() for i in user['interests']])
            
            user_text = ' '.join(features)
            user_texts.append(user_text)
            user_ids.append(user.get('id', len(user_ids)))
        
        if self.use_sbert:
            self.user_vectors = np.array(self.sbert_model.encode(user_texts))
        else:
            if fit_vectorizer:
                self.user_vectors = self.tfidf_vectorizer.fit_transform(user_texts).toarray()
            else:
                self.user_vectors = self.tfidf_vectorizer.transform(user_texts).toarray()
        
        self.user_ids = np.array(user_ids)
        print(f"Created {len(user_texts)} user vectors")
    
    def prepare_job_features(self, jobs_df):
        """Prepare job feature vectors"""
        print("Preparing job features...")
        
        job_texts = []
        job_ids = []
        
        for _, job in jobs_df.iterrows():
            features = []
            
            if 'title' in job and pd.notna(job['title']):
                features.append(str(job['title']).lower())
            
            if 'description_cleaned' in job and pd.notna(job['description_cleaned']):
                features.append(str(job['description_cleaned']).lower())
            elif 'description' in job and pd.notna(job['description']):
                features.append(str(job['description']).lower())
            
            if 'skills_cleaned' in job:
                if isinstance(job['skills_cleaned'], list):
                    features.extend([str(s).lower() for s in job['skills_cleaned']])
                else:
                    features.append(str(job['skills_cleaned']).lower())
            
            job_text = ' '.join(features)
            job_texts.append(job_text)
            job_ids.append(job.get('id', len(job_ids)))
        
        if self.use_sbert:
            self.job_vectors = np.array(self.sbert_model.encode(job_texts))
        else:
            if hasattr(self.tfidf_vectorizer, 'vocabulary_') and len(self.tfidf_vectorizer.vocabulary_) > 0:
                self.job_vectors = self.tfidf_vectorizer.transform(job_texts).toarray()
            else:
                self.job_vectors = self.tfidf_vectorizer.fit_transform(job_texts).toarray()
        
        self.job_ids = np.array(job_ids)
        print(f"Created {len(job_texts)} job vectors")
    
    def prepare_role_features(self, roles_df):
        """Prepare career role feature vectors"""
        print("Preparing role features...")
        
        role_texts = []
        role_ids = []
        
        for _, role in roles_df.iterrows():
            features = []
            
            if 'title' in role and pd.notna(role['title']):
                features.append(str(role['title']).lower())
            
            if 'description' in role and pd.notna(role['description']):
                features.append(str(role['description']).lower())
            
            if 'required_skills' in role:
                if isinstance(role['required_skills'], list):
                    features.extend([str(s).lower() for s in role['required_skills']])
                else:
                    features.append(str(role['required_skills']).lower())
            
            role_text = ' '.join(features)
            role_texts.append(role_text)
            role_ids.append(role.get('id', len(role_ids)))
        
        if self.use_sbert:
            self.role_vectors = np.array(self.sbert_model.encode(role_texts))
        else:
            if hasattr(self.tfidf_vectorizer, 'vocabulary_') and len(self.tfidf_vectorizer.vocabulary_) > 0:
                self.role_vectors = self.tfidf_vectorizer.transform(role_texts).toarray()
            else:
                self.role_vectors = self.tfidf_vectorizer.fit_transform(role_texts).toarray()
        
        self.role_ids = np.array(role_ids)
        print(f"Created {len(role_texts)} role vectors")
    
    def save(self, model_dir='ml/models'):
        """Save the model"""
        model_path = Path(model_dir)
        model_path.mkdir(parents=True, exist_ok=True)
        
        if not self.use_sbert:
            joblib.dump(self.tfidf_vectorizer, model_path / 'tfidf_vectorizer.joblib')
        
        np.save(model_path / 'user_vectors.npy', self.user_vectors)
        np.save(model_path / 'job_vectors.npy', self.job_vectors)
        np.save(model_path / 'role_vectors.npy', self.role_vectors)
        np.save(model_path / 'user_ids.npy', self.user_ids)
        np.save(model_path / 'job_ids.npy', self.job_ids)
        np.save(model_path / 'role_ids.npy', self.role_ids)
        
        metadata = {
            'use_sbert': self.use_sbert,
            'num_users': len(self.user_ids) if self.user_ids is not None else 0,
            'num_jobs': len(self.job_ids) if self.job_ids is not None else 0,
            'num_roles': len(self.role_ids) if self.role_ids is not None else 0
        }
        
        with open(model_path / 'content_based_metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✅ Model saved to {model_path}")

print("✅ ContentBasedRecommender class defined!")


In [ ]:
# Train Content-Based Model
print("="*70)
print("TRAINING CONTENT-BASED RECOMMENDATION MODEL")
print("="*70)

data_dir = Path('data/processed')

# Load processed data or create sample data
users_df = pd.read_csv(data_dir / 'resumes_processed.csv') if (data_dir / 'resumes_processed.csv').exists() else None
jobs_df = pd.read_csv(data_dir / 'jobs_processed.csv') if (data_dir / 'jobs_processed.csv').exists() else None

if users_df is None or len(users_df) == 0:
    print("No user data found, creating sample data...")
    users_df = pd.DataFrame({
        'id': range(100),
        'skills_cleaned': [['python', 'sql', 'machine learning']] * 100,
        'experience_years': np.random.randint(0, 10, 100),
        'current_role': ['Data Scientist', 'Software Engineer', 'ML Engineer'] * 33 + ['Data Scientist']
    })

if jobs_df is None or len(jobs_df) == 0:
    print("No job data found, creating sample data...")
    jobs_df = pd.DataFrame({
        'id': range(50),
        'title': ['Senior Data Scientist', 'Python Developer', 'ML Engineer'] * 16 + ['Senior Data Scientist', 'Python Developer'],
        'description': ['Looking for experienced data scientist'] * 50,
        'skills_cleaned': [['python', 'machine learning', 'sql']] * 50
    })

# Create roles data
roles_df = pd.DataFrame({
    'id': range(20),
    'title': [
        'Data Scientist', 'Software Engineer', 'ML Engineer', 'Data Analyst',
        'Backend Developer', 'Frontend Developer', 'Full Stack Developer',
        'DevOps Engineer', 'Cloud Architect', 'Product Manager',
        'UX Designer', 'Data Engineer', 'Security Engineer', 'QA Engineer',
        'Mobile Developer', 'Blockchain Developer', 'AI Researcher',
        'Business Analyst', 'Technical Writer', 'Solutions Architect'
    ],
    'description': ['Career role description'] * 20,
    'required_skills': [['python', 'sql'], ['java', 'spring'], ['python', 'tensorflow']] * 6 + [['python', 'sql'], ['java', 'spring']]
})

# Train model
model = ContentBasedRecommender(use_sbert=False)

# Fit vectorizer on combined corpus
print("Fitting vectorizer on combined corpus...")
all_texts = []

if users_df is not None and len(users_df) > 0:
    for _, user in users_df.iterrows():
        features = []
        if 'skills_cleaned' in user and isinstance(user['skills_cleaned'], list):
            features.extend(user['skills_cleaned'])
        elif 'skills' in user:
            features.extend(str(user['skills']).lower().split())
        if 'experience_years' in user:
            features.append(f"{int(user['experience_years'])} years experience")
        if 'current_role' in user and pd.notna(user['current_role']):
            features.append(str(user['current_role']).lower())
        all_texts.append(' '.join(features))

if jobs_df is not None and len(jobs_df) > 0:
    for _, job in jobs_df.iterrows():
        features = []
        if 'title' in job and pd.notna(job['title']):
            features.append(str(job['title']).lower())
        if 'description_cleaned' in job and pd.notna(job['description_cleaned']):
            features.append(str(job['description_cleaned']).lower())
        elif 'description' in job and pd.notna(job['description']):
            features.append(str(job['description']).lower())
        if 'skills_cleaned' in job:
            if isinstance(job['skills_cleaned'], list):
                features.extend([str(s).lower() for s in job['skills_cleaned']])
            else:
                features.append(str(job['skills_cleaned']).lower())
        all_texts.append(' '.join(features))

for _, role in roles_df.iterrows():
    features = []
    if 'title' in role and pd.notna(role['title']):
        features.append(str(role['title']).lower())
    if 'description' in role and pd.notna(role['description']):
        features.append(str(role['description']).lower())
    if 'required_skills' in role:
        if isinstance(role['required_skills'], list):
            features.extend([str(s).lower() for s in role['required_skills']])
        else:
            features.append(str(role['required_skills']).lower())
    all_texts.append(' '.join(features))

if len(all_texts) > 0:
    model.tfidf_vectorizer.fit(all_texts)
    print(f"Vectorizer fitted on {len(all_texts)} combined documents")

if users_df is not None and len(users_df) > 0:
    model.prepare_user_features(users_df, fit_vectorizer=False)

if jobs_df is not None and len(jobs_df) > 0:
    model.prepare_job_features(jobs_df)

model.prepare_role_features(roles_df)
model.save()

print("✅ Content-based model training complete!")


In [ ]:
# Collaborative Filtering Model Class
class CollaborativeFilteringModel:
    """Collaborative filtering using SVD"""
    
    def __init__(self, n_components=50, use_surprise=True):
        self.n_components = n_components
        self.use_surprise = use_surprise and SURPRISE_AVAILABLE
        
        if self.use_surprise:
            self.model = SurpriseSVD(n_factors=n_components, random_state=42)
        else:
            self.model = TruncatedSVD(n_components=n_components, random_state=42)
            self.scaler = StandardScaler()
        
        self.user_mapping = {}
        self.item_mapping = {}
        self.reverse_user_mapping = {}
        self.reverse_item_mapping = {}
        self.interaction_matrix = None
        self.test_interactions = None
        self.is_trained = False
    
    def prepare_interaction_matrix(self, interactions_df):
        """Create user-item interaction matrix"""
        print("Preparing interaction matrix...")
        
        unique_users = interactions_df['user_id'].unique()
        unique_items = interactions_df['item_id'].unique()
        
        self.user_mapping = {user_id: idx for idx, user_id in enumerate(unique_users)}
        self.item_mapping = {item_id: idx for idx, item_id in enumerate(unique_items)}
        self.reverse_user_mapping = {idx: user_id for user_id, idx in self.user_mapping.items()}
        self.reverse_item_mapping = {idx: item_id for item_id, idx in self.item_mapping.items()}
        
        n_users = len(unique_users)
        n_items = len(unique_items)
        self.interaction_matrix = np.zeros((n_users, n_items))
        
        for _, row in interactions_df.iterrows():
            user_idx = self.user_mapping[row['user_id']]
            item_idx = self.item_mapping[row['item_id']]
            rating = row.get('rating', 1)
            self.interaction_matrix[user_idx, item_idx] = rating
        
        print(f"Created interaction matrix: {n_users} users × {n_items} items")
        print(f"Sparsity: {(self.interaction_matrix == 0).sum() / (n_users * n_items) * 100:.2f}%")
    
    def train(self, interactions_df, test_size=0.2):
        """Train the SVD model"""
        print("Training collaborative filtering model...")
        
        if self.use_surprise:
            reader = Reader(rating_scale=(0, 5))
            data = Dataset.load_from_df(interactions_df[['user_id', 'item_id', 'rating']], reader)
            trainset, testset = surprise_train_test_split(data, test_size=test_size, random_state=42)
            self.model.fit(trainset)
            
            predictions = self.model.test(testset)
            rmse = accuracy.rmse(predictions, verbose=False)
            mae = accuracy.mae(predictions, verbose=False)
            print(f"Test RMSE: {rmse:.4f}, Test MAE: {mae:.4f}")
        else:
            train_interactions, test_interactions = train_test_split(
                interactions_df, test_size=test_size, random_state=42
            )
            
            print(f"Training on {len(train_interactions)} interactions, testing on {len(test_interactions)} interactions")
            
            if self.interaction_matrix is None:
                self.prepare_interaction_matrix(train_interactions)
            
            self.test_interactions = test_interactions
            self.interaction_matrix = self.scaler.fit_transform(self.interaction_matrix.T).T
            self.model.fit(self.interaction_matrix)
            
            if len(test_interactions) > 0:
                test_errors = []
                for _, row in test_interactions.iterrows():
                    user_id = row['user_id']
                    item_id = row['item_id']
                    true_rating = row['rating']
                    
                    if user_id in self.user_mapping and item_id in self.item_mapping:
                        pred_rating = self.predict(user_id, item_id)
                        test_errors.append(abs(pred_rating - true_rating))
                
                if test_errors:
                    mae = np.mean(test_errors)
                    rmse = np.sqrt(np.mean([e**2 for e in test_errors]))
                    print(f"Test MAE: {mae:.4f}, Test RMSE: {rmse:.4f}")
        
        self.is_trained = True
        print("Model training complete!")
    
    def predict(self, user_id, item_id):
        """Predict rating for user-item pair"""
        if not self.is_trained:
            raise ValueError("Model not trained yet")
        
        if self.use_surprise:
            try:
                prediction = self.model.predict(user_id, item_id)
                return prediction.est
            except:
                return 0.0
        else:
            if user_id not in self.user_mapping or item_id not in self.item_mapping:
                return 0.0
            
            user_idx = self.user_mapping[user_id]
            item_idx = self.item_mapping[item_id]
            
            user_factors = self.model.transform(self.interaction_matrix[user_idx:user_idx+1])[0]
            item_factors = self.model.components_[:, item_idx]
            
            prediction = np.dot(user_factors, item_factors)
            return float(prediction)
    
    def save(self, model_dir='ml/models'):
        """Save the model"""
        model_path = Path(model_dir)
        model_path.mkdir(parents=True, exist_ok=True)
        
        joblib.dump(self.model, model_path / 'svd_model.joblib')
        
        if not self.use_surprise:
            joblib.dump(self.scaler, model_path / 'svd_scaler.joblib')
            if self.interaction_matrix is not None:
                np.save(model_path / 'interaction_matrix.npy', self.interaction_matrix)
        
        with open(model_path / 'svd_user_mapping.json', 'w') as f:
            json.dump({str(k): v for k, v in self.user_mapping.items()}, f)
        
        with open(model_path / 'svd_item_mapping.json', 'w') as f:
            json.dump({str(k): v for k, v in self.item_mapping.items()}, f)
        
        metadata = {
            'n_components': self.n_components,
            'use_surprise': self.use_surprise,
            'n_users': len(self.user_mapping),
            'n_items': len(self.item_mapping),
            'is_trained': self.is_trained
        }
        
        with open(model_path / 'svd_metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✅ Model saved to {model_path}")

print("✅ CollaborativeFilteringModel class defined!")


In [ ]:
# Train Collaborative Filtering Model
print("="*70)
print("TRAINING COLLABORATIVE FILTERING MODEL")
print("="*70)

data_dir = Path('data/processed')

# Load interactions data or create sample data
interactions_df = pd.read_csv(data_dir / 'interactions_processed.csv') if (data_dir / 'interactions_processed.csv').exists() else None

if interactions_df is None or len(interactions_df) == 0:
    print("No interactions data found, creating sample data...")
    n_users = 100
    n_items = 50
    n_interactions = 500
    
    interactions_df = pd.DataFrame({
        'user_id': np.random.randint(0, n_users, n_interactions),
        'item_id': np.random.randint(0, n_items, n_interactions),
        'rating': np.random.randint(1, 6, n_interactions)
    })
    
    interactions_df = interactions_df.drop_duplicates(subset=['user_id', 'item_id'])

# Train model
model = CollaborativeFilteringModel(n_components=50, use_surprise=SURPRISE_AVAILABLE)
model.train(interactions_df)
model.save()

print("✅ Collaborative filtering model training complete!")


## Step 5: Career Role Classifier


In [ ]:
# Career Role Classifier Class
class CareerRoleClassifier:
    """Supervised classifier for predicting suitable career roles"""
    
    def __init__(self, model_type='random_forest', n_estimators=200, max_depth=15, min_samples_split=10):
        self.model_type = model_type
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        
        if model_type == 'random_forest':
            self.model = RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_split=min_samples_split,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1,
                class_weight='balanced',
                max_features='sqrt',
                bootstrap=True,
                oob_score=True,
                max_samples=0.8
            )
        else:
            self.model = DecisionTreeClassifier(
                max_depth=max_depth,
                min_samples_split=min_samples_split,
                random_state=42,
                class_weight='balanced'
            )
        
        self.label_encoder = LabelEncoder()
        self.feature_names = []
        self.is_trained = False
    
    def prepare_features(self, users_df, roles_df=None):
        """Prepare features from user data"""
        print("Preparing features...")
        
        features_list = []
        labels_list = []
        
        for _, user in users_df.iterrows():
            feature_dict = {}
            feature_dict['experience_years'] = user.get('experience_years', 0)
            
            skills = user.get('skills_cleaned', [])
            if isinstance(skills, str):
                try:
                    skills = ast.literal_eval(skills)
                except:
                    skills = [skills]
            
            if not isinstance(skills, list):
                skills = []
            
            common_skills = [
                'python', 'java', 'javascript', 'sql', 'react', 'node.js',
                'machine learning', 'data science', 'aws', 'docker',
                'git', 'linux', 'html', 'css', 'mongodb', 'tensorflow',
                'pytorch', 'agile', 'scrum', 'rest api', 'spring', 'pandas',
                'numpy', 'statistics', 'microservices', 'kubernetes', 'ci/cd',
                'typescript', 'design', 'figma', 'user research', 'prototyping',
                'product management', 'analytics', 'jira', 'confluence'
            ]
            
            for skill in common_skills:
                skill_found = any(
                    skill.lower() in str(s).lower() or str(s).lower() in skill.lower()
                    for s in skills
                )
                feature_dict[f'skill_{skill}'] = 1 if skill_found else 0
            
            feature_dict['num_skills'] = len(skills)
            
            data_skills = ['python', 'sql', 'machine learning', 'data science', 'pandas', 'numpy', 'statistics']
            web_skills = ['javascript', 'react', 'node.js', 'html', 'css', 'typescript']
            backend_skills = ['java', 'spring', 'sql', 'rest api', 'microservices']
            devops_skills = ['docker', 'kubernetes', 'aws', 'linux', 'ci/cd']
            design_skills = ['design', 'figma', 'user research', 'prototyping']
            pm_skills = ['agile', 'scrum', 'product management', 'analytics', 'jira']
            
            feature_dict['data_skill_count'] = sum(1 for s in skills if any(ds in str(s).lower() for ds in data_skills))
            feature_dict['web_skill_count'] = sum(1 for s in skills if any(ws in str(s).lower() for ws in web_skills))
            feature_dict['backend_skill_count'] = sum(1 for s in skills if any(bs in str(s).lower() for bs in backend_skills))
            feature_dict['devops_skill_count'] = sum(1 for s in skills if any(ds in str(s).lower() for ds in devops_skills))
            feature_dict['design_skill_count'] = sum(1 for s in skills if any(ds in str(s).lower() for ds in design_skills))
            feature_dict['pm_skill_count'] = sum(1 for s in skills if any(ps in str(s).lower() for ps in pm_skills))
            
            education = user.get('education', [])
            if isinstance(education, str):
                try:
                    education = ast.literal_eval(education)
                except:
                    education = []
            
            feature_dict['has_bachelor'] = 1 if any('bachelor' in str(e).lower() for e in education) else 0
            feature_dict['has_master'] = 1 if any('master' in str(e).lower() for e in education) else 0
            feature_dict['has_phd'] = 1 if any('phd' in str(e).lower() or 'doctorate' in str(e).lower() for e in education) else 0
            
            features_list.append(feature_dict)
            
            if 'target_role' in user and pd.notna(user['target_role']):
                labels_list.append(str(user['target_role']))
            elif 'role_normalized' in user and pd.notna(user['role_normalized']):
                labels_list.append(str(user['role_normalized']))
            elif 'current_role' in user and pd.notna(user['current_role']):
                labels_list.append(str(user['current_role']))
            else:
                labels_list.append('Unknown')
        
        features_df = pd.DataFrame(features_list)
        self.feature_names = features_df.columns.tolist()
        
        self.label_encoder.fit(labels_list)
        encoded_labels = self.label_encoder.transform(labels_list)
        
        return features_df.values, encoded_labels
    
    def train(self, users_df, roles_df=None, test_size=0.2):
        """Train the classifier"""
        print("Training classifier...")
        
        X, y = self.prepare_features(users_df, roles_df)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
        
        train_counts = Counter(y_train)
        test_counts = Counter(y_test)
        print(f"Training samples per class: {dict(train_counts)}")
        print(f"Test samples per class: {dict(test_counts)}")
        
        print(f"Training {self.model_type} on {len(X_train)} samples...")
        self.model.fit(X_train, y_train)
        
        y_pred = self.model.predict(X_test)
        cv_scores = cross_val_score(self.model, X_train, y_train, cv=5, scoring='accuracy')
        print(f"\nCross-validation accuracy (5-fold): {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
        
        y_train_pred = self.model.predict(X_train)
        train_accuracy = accuracy_score(y_train, y_train_pred)
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        
        print(f"\nModel Performance:")
        print(f"Train Accuracy: {train_accuracy:.4f}")
        print(f"Test Accuracy: {accuracy:.4f}")
        print(f"Overfitting Gap: {train_accuracy - accuracy:.4f} (should be < 0.15)")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")
        
        if train_accuracy - accuracy > 0.15:
            print("\n⚠️  WARNING: Potential overfitting detected!")
        
        if hasattr(self.model, 'feature_importances_'):
            feature_importance = dict(zip(self.feature_names, self.model.feature_importances_))
            top_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)[:10]
            print(f"\nTop 10 Important Features:")
            for feature, importance in top_features:
                print(f"  {feature}: {importance:.4f}")
        
        self.is_trained = True
        
        return {
            'accuracy': float(accuracy),
            'precision': float(precision),
            'recall': float(recall),
            'f1_score': float(f1)
        }
    
    def save(self, model_dir='ml/models'):
        """Save the model"""
        model_path = Path(model_dir)
        model_path.mkdir(parents=True, exist_ok=True)
        
        joblib.dump(self.model, model_path / 'career_classifier.joblib')
        joblib.dump(self.label_encoder, model_path / 'label_encoder.joblib')
        
        metadata = {
            'model_type': self.model_type,
            'n_estimators': self.n_estimators if self.model_type == 'random_forest' else None,
            'max_depth': self.max_depth,
            'feature_names': self.feature_names,
            'n_classes': len(self.label_encoder.classes_),
            'classes': self.label_encoder.classes_.tolist(),
            'is_trained': self.is_trained
        }
        
        with open(model_path / 'classifier_metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✅ Model saved to {model_path}")

print("✅ CareerRoleClassifier class defined!")


In [ ]:
# Train Career Role Classifier
print("="*70)
print("TRAINING CAREER ROLE CLASSIFIER")
print("="*70)

data_dir = Path('data/processed')

# Load user data (from Kaggle datasets)
users_df = pd.read_csv(data_dir / 'resumes_processed.csv')
print(f"Loaded {len(users_df)} user profiles for training")

# Train model
classifier = CareerRoleClassifier(model_type='random_forest', n_estimators=200, max_depth=15, min_samples_split=10)
metrics = classifier.train(users_df)
classifier.save()

print("✅ Career role classifier training complete!")

## Step 6: Chatbot Intent Classifier


In [ ]:
# Chatbot Intent Classifier Class
class ChatbotIntentClassifier:
    """Intent classifier for chatbot"""
    
    def __init__(self, model_type='logistic_regression'):
        self.model_type = model_type
        self.vectorizer = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words='english',
            min_df=1,
            max_df=0.9
        )
        
        if model_type == 'logistic_regression':
            self.model = LogisticRegression(
                max_iter=2000,
                C=1.0,
                random_state=42,
                solver='lbfgs',
                multi_class='multinomial'
            )
        else:
            self.model = SVC(kernel='linear', random_state=42)
        
        self.label_encoder = {}
        self.reverse_label_encoder = {}
        self.is_trained = False
    
    def create_training_data(self):
        """Create training data for intent classification"""
        intents = {
            'career_advice': [
                'what career should i choose', 'which career is best for me', 'career guidance',
                'help me choose a career', 'what job should i do', 'career recommendations',
                'suggest a career path', 'what are good careers', 'career options for me',
                'what career path should i take', 'recommend a career', 'best career for my skills',
                'career advice please', 'what career suits me', 'career suggestions',
                'guide me on career', 'help with career choice', 'career direction',
                'what should i do for career', 'career planning help', 'career decision',
                'which field should i choose', 'career recommendation', 'suitable career for me'
            ],
            'job_search': [
                'find me a job', 'show available jobs', 'job openings', 'search jobs',
                'find jobs near me', 'job opportunities', 'where can i find work',
                'job listings', 'available positions', 'hiring jobs', 'jobs available',
                'show me jobs', 'job search', 'find employment', 'job vacancies',
                'open positions', 'job postings', 'available jobs', 'find work',
                'job market', 'employment opportunities', 'job board', 'career opportunities',
                'job hunting', 'looking for job', 'need a job', 'job openings near me'
            ],
            'skill_gap': [
                'what skills do i need', 'skill gap analysis', 'what am i missing',
                'skills required for this role', 'what do i need to learn', 'missing skills',
                'skill requirements', 'what skills should i have', 'check my skills',
                'analyze my skills', 'what skills am i lacking', 'skill assessment',
                'required skills', 'skills needed', 'what skills do i lack',
                'skill gap check', 'missing competencies', 'skills gap', 'skill analysis',
                'what skills required', 'check skill gaps', 'identify missing skills',
                'skill requirements for role', 'what skills missing', 'skill evaluation'
            ],
            'learning_path': [
                'how to learn', 'learning resources', 'courses to take', 'what should i study',
                'learning path', 'training resources', 'how to improve', 'study materials',
                'recommended courses', 'where to learn', 'how can i learn', 'learning guide',
                'training path', 'study plan', 'courses recommended', 'how to study',
                'learning materials', 'educational resources', 'training courses',
                'how to develop skills', 'skill development', 'learning roadmap',
                'study resources', 'training plan', 'how to gain skills', 'learning strategy'
            ],
            'recommendation_explanation': [
                'why this recommendation', 'explain recommendation', 'why am i seeing this',
                'how was this recommended', 'reason for suggestion', 'why this job',
                'explain the match', 'why is this recommended', 'explain why',
                'why did you recommend', 'reason for recommendation', 'how was this chosen',
                'explain the suggestion', 'why this match', 'explain job match',
                'why recommend this', 'explanation please', 'why this role',
                'how did you decide', 'reason behind recommendation', 'explain the choice',
                'why is this suitable', 'explain suitability', 'why matched'
            ],
            'general_info': [
                'hello', 'hi', 'help', 'hey', 'greetings', 'good morning', 'good afternoon',
                'good evening', 'how are you', 'what can you do', 'tell me about yourself',
                'what is this', 'introduction', 'who are you', 'what are you',
                'help me', 'assist me', 'support', 'information', 'tell me more',
                'explain', 'what is', 'how does this work', 'guide me', 'instructions',
                'what is this system', 'about', 'thanks', 'thank you'
            ]
        }
        
        texts = []
        labels = []
        
        for intent, queries in intents.items():
            for query in queries:
                texts.append(query)
                labels.append(intent)
        
        variations = {
            'career_advice': ['career', 'job', 'role', 'profession', 'occupation'],
            'job_search': ['job', 'position', 'opening', 'vacancy', 'hiring'],
            'skill_gap': ['skill', 'ability', 'competency', 'expertise'],
            'learning_path': ['learn', 'study', 'course', 'training', 'education'],
            'recommendation_explanation': ['why', 'explain', 'reason', 'how'],
            'general_info': ['hello', 'hi', 'help', 'thanks']
        }
        
        for intent, keywords in variations.items():
            for keyword in keywords:
                texts.append(f"i need {keyword}")
                labels.append(intent)
                texts.append(f"tell me about {keyword}")
                labels.append(intent)
                texts.append(f"help with {keyword}")
                labels.append(intent)
                texts.append(f"show me {keyword}")
                labels.append(intent)
                texts.append(f"i want {keyword}")
                labels.append(intent)
        
        augmentations = {
            'career_advice': [
                'i need career help', 'career guidance needed', 'what career path',
                'suggest career', 'career options', 'best career choice',
                'career recommendation needed', 'help choose career', 'career advice needed'
            ],
            'job_search': [
                'i need a job', 'find employment', 'job opportunities available',
                'show job openings', 'available positions', 'hiring now',
                'job search help', 'find work', 'employment search'
            ],
            'skill_gap': [
                'what skills required', 'skills i need', 'required competencies',
                'skill assessment needed', 'check my skills', 'skill evaluation',
                'missing abilities', 'what do i need to know', 'skill requirements check'
            ],
            'learning_path': [
                'how to study', 'learning guide', 'training needed',
                'courses available', 'study resources', 'how to improve skills',
                'education path', 'skill development', 'training resources'
            ],
            'recommendation_explanation': [
                'why recommend', 'explain why', 'reason for this',
                'how was this chosen', 'why this match', 'explain the choice',
                'why suitable', 'explain match', 'reason behind'
            ],
            'general_info': [
                'hi there', 'greetings', 'can you help', 'what is this',
                'how does it work', 'tell me more', 'i need help',
                'information please', 'explain system', 'what can you do'
            ]
        }
        
        for intent, augs in augmentations.items():
            for aug in augs:
                texts.append(aug)
                labels.append(intent)
        
        return texts, labels
    
    def train(self, texts=None, labels=None):
        """Train the intent classifier"""
        print("Training chatbot intent classifier...")
        
        if texts is None or labels is None:
            texts, labels = self.create_training_data()
        
        unique_labels = list(set(labels))
        self.label_encoder = {label: idx for idx, label in enumerate(unique_labels)}
        self.reverse_label_encoder = {idx: label for label, idx in self.label_encoder.items()}
        
        encoded_labels = [self.label_encoder[label] for label in labels]
        
        X = self.vectorizer.fit_transform(texts)
        y = np.array(encoded_labels)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        n_train = X_train.shape[0] if hasattr(X_train, 'shape') else len(X_train)
        print(f"Training {self.model_type} on {n_train} samples...")
        self.model.fit(X_train, y_train)
        
        y_pred = self.model.predict(X_test)
        y_train_pred = self.model.predict(X_train)
        
        train_accuracy = accuracy_score(y_train, y_train_pred)
        test_accuracy = accuracy_score(y_test, y_pred)
        
        cv_scores = cross_val_score(self.model, X_train, y_train, cv=5, scoring='accuracy')
        print(f"\nCross-validation accuracy (5-fold): {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
        
        print(f"\nModel Performance:")
        print(f"Train Accuracy: {train_accuracy:.4f}")
        print(f"Test Accuracy: {test_accuracy:.4f}")
        print(f"Overfitting Gap: {train_accuracy - test_accuracy:.4f} (should be < 0.15)")
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred, target_names=unique_labels))
        
        if train_accuracy - test_accuracy > 0.15:
            print("\n⚠️  WARNING: Potential overfitting detected!")
        
        self.is_trained = True
        
        return test_accuracy
    
    def save(self, model_dir='ml/models'):
        """Save the model"""
        model_path = Path(model_dir)
        model_path.mkdir(parents=True, exist_ok=True)
        
        joblib.dump(self.model, model_path / 'intent_classifier.joblib')
        joblib.dump(self.vectorizer, model_path / 'intent_vectorizer.joblib')
        
        with open(model_path / 'intent_label_encoder.json', 'w') as f:
            json.dump(self.label_encoder, f, indent=2)
        
        with open(model_path / 'intent_reverse_encoder.json', 'w') as f:
            json.dump(self.reverse_label_encoder, f, indent=2)
        
        metadata = {
            'model_type': self.model_type,
            'n_intents': len(self.label_encoder),
            'intents': list(self.label_encoder.keys()),
            'is_trained': self.is_trained
        }
        
        with open(model_path / 'intent_metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✅ Model saved to {model_path}")

print("✅ ChatbotIntentClassifier class defined!")


In [ ]:
# Train Chatbot Intent Classifier
print("="*70)
print("TRAINING CHATBOT INTENT CLASSIFIER")
print("="*70)

classifier = ChatbotIntentClassifier(model_type='logistic_regression')
accuracy = classifier.train()

# Test with sample queries
test_queries = [
    "what career should I choose?",
    "find me a job",
    "what skills do I need?",
    "how can I learn Python?",
    "why is this job recommended?",
    "hello"
]

print("\nTesting with sample queries:")
for query in test_queries:
    X = classifier.vectorizer.transform([query])
    prediction = classifier.model.predict(X)[0]
    probabilities = classifier.model.predict_proba(X)[0]
    intent = classifier.reverse_label_encoder[prediction]
    confidence = probabilities[prediction]
    print(f"Query: '{query}'")
    print(f"Intent: {intent} (confidence: {confidence:.4f})")
    print()

classifier.save()

print("✅ Intent classifier training complete!")


## Step 7: Download Models

All models have been trained and saved! You can now download them.


In [ ]:
# List all trained models
model_path = Path('ml/models')
if model_path.exists():
    print("📦 Trained Models:")
    print("="*70)
    total_size = 0
    for file in sorted(model_path.glob('*')):
        size = file.stat().st_size / 1024  # Size in KB
        total_size += size
        print(f"  {file.name:50s} {size:8.2f} KB")
    print("="*70)
    print(f"Total size: {total_size/1024:.2f} MB")
else:
    print("❌ No models directory found")


In [ ]:
# Create a zip file of all models for easy download
import shutil

if Path('ml/models').exists():
    shutil.make_archive('trained_models', 'zip', 'ml/models')
    zip_size = Path('trained_models.zip').stat().st_size / (1024*1024)
    print(f"✅ Created trained_models.zip ({zip_size:.2f} MB)")
    print("\n📥 To download:")
    print("   1. In Colab, go to Files panel (left sidebar)")
    print("   2. Right-click on 'trained_models.zip'")
    print("   3. Select 'Download'")
    print("\n   OR use the code below to download directly:")
    print("   from google.colab import files")
    print("   files.download('trained_models.zip')")
else:
    print("❌ Models directory not found")


## 🎉 Training Complete!

All models have been successfully trained:
- ✅ Content-Based Recommendation Model
- ✅ Collaborative Filtering Model (SVD)
- ✅ Career Role Classifier
- ✅ Chatbot Intent Classifier

**Next Steps:**
1. Download the `trained_models.zip` file
2. Extract it to your project's `backend/ml/models/` directory
3. Your models are ready to use!
